## Intro

In the **MCMC** methods, such as Metropolis and Gibbs, the objective was exploration: generating random samples to reconstruct an entire probability distribution $\pi(\mathbf{x})$, compute expectations, or characterize posterior uncertainty.

**Deterministic Optimization** changes the objective entirely. We no longer want to explore the whole terrain or generate samples - we want to find the single best point. Given an objective function (often called an energy function, cost function, or loss) $f: \mathbb{R}^D \to \mathbb{R}$, the goal is to compute:$$\mathbf{x}^* = \arg\min_{\mathbf{x} \in \mathbb{R}^D} f(\mathbf{x}) \quad \left(\text{or } \arg\max_{\mathbf{x}} f(\mathbf{x})\right)$$

**Deterministic** because:
- *No random variables or dice rolls are involved*: Given a starting location $\mathbf{x}_0$, the sequence of steps $\mathbf{x}_0 \to \mathbf{x}_1 \to \dots \to \mathbf{x}^*$ is completely repeatable and reproducible.
- *Decisions are greedy/mechanistic*: At each step, the algorithm inspects local geometric properties (such as slopes and curvatures) and moves deterministically toward lower energy.
- Most deterministic algorithms can only guarantee finding a local optimum. Unless, the function is **convex**.

### Note on Taylor Aprroximations

Almost all classical unconstrained deterministic optimization algorithms work by approximating the complicated true function $f(\mathbf{x})$ around the current point $\mathbf{x}_k$ using a truncated Taylor expansion: $$f(\mathbf{x}_k + \Delta\mathbf{x}) \approx f(\mathbf{x}_k) + \nabla f(\mathbf{x}_k)^T \Delta\mathbf{x} + \frac{1}{2} \Delta\mathbf{x}^T \nabla^2 f(\mathbf{x}_k) \Delta\mathbf{x}$$

where:
- **The Gradient Vector $\mathbf{g}_k = \nabla f(\mathbf{x}_k) \in \mathbb{R}^D$**: Represents the first derivative (slope). It points in the direction of steepest ascent; therefore, $-\nabla f(\mathbf{x}_k)$ points in the direction of steepest descent.
- **The Hessian Matrix $\mathbf{H}_k = \nabla^2 f(\mathbf{x}_k) \in \mathbb{R}^{D \times D}$**: Represents the second derivatives (curvature). It measures how the slope changes in different directions (bowl vs. saddle vs. valley).

### Note of convexity

For convex functions: local optimum = global optimum. A function $f: \mathbb{R}^n \rarr \mathbb{R}$ is convex if for any $x, y; x,y \in Q$ and any $\lambda \in [0,1]$:
$$f(\lambda x + (1-\lambda)y) \leq \lambda f(y) + (1-\lambda) f(x)$$
If $f(x), \nabla f(x), \nabla^2 f(x)$ exist, the function $f(x)$ is convex is $\nabla^2 f(x)>0; \forall x \in Q$. $\nabla^2 f(x)$ is the Hessian matrix of $f$. By definituin, the matrix of the second deruvatives is positive definite ehan all the eigenvalues are positive. $$\forall \lambda; \lambda^+ H \lambda > 0$$



## Newton Method
Newton’s method is a Second-Order Optimization Algorithm.

First-order methods like Steepest Descent look at the slope under your feet and take a cautious step downhill. They assume the world is flat (a plane) locally.

**Newton's approach assumes the world is curved (a bowl).**

Instead of approximating the function with a flat tangent line, Newton's method approximates the objective function with a quadratic model (a paraboloid) using both the slope (gradient) and the curvature (Hessian). It then calculates the exact bottom of that quadratic bowl and jumps directly there in a single step.

At your current position $x_k$, you evaluate three numbers:
1. Current elevation: $f(x_k)$
2. Current slope: $f'(x_k)$
3. Current curvature: $f''(x_k)$
You approximate $f(x)$ with a 1D parabola $m(x)$ centered at $x_k$: $$m(x) = f(x_k) + f'(x_k)(x - x_k) + \frac{1}{2} f''(x_k)(x - x_k)^2$$

To find the minimum of this surrogate parabola, differentiate with respect to $x$ and set the result to zero: $$\frac{d}{dx}m(x) = f'(x_k) + f''(x_k)(x - x_k) = 0$$
Solve directly for $x$: $$f''(x_k)(x - x_k) = -f'(x_k) \implies x - x_k = -\frac{f'(x_k)}{f''(x_k)}$$
This yields the **1D Newton update formula**: $$x_{k+1} = x_k - \frac{f'(x_k)}{f''(x_k)}$$

### Intuition: Why Divide the Slope by the Curvature?
The update is: $$\Delta x = -\frac{\text{Velocity}}{\text{Acceleration}} = -\frac{\text{Slope}}{\text{Rate of change of slope}}$$
- Slope $f'(x_k)$ tells you the direction and steepness: If $f'(x_k) > 0$, the function slopes upward to the right, so the minimum lies to the left ($\Delta x < 0$).
- Curvature $f''(x_k)$ acts as an automatic, adaptive step-size regulator:
    - Sharp bowl ($f''(x_k)$ is large): The walls rise steeply. If you take a large step, you will blow past the bottom. The large denominator scales your step size down to keep you in the bowl.
    - Flat, shallow valley ($f''(x_k)$ is tiny): The slope changes very slowly over long distances. The minimum is likely far away. Dividing by a tiny number makes the step large, accelerating you across the plateau.

Unlike Gradient Descent, where you must guess and tune a learning rate $\alpha$, 1D Newton's method calculates its own step size automatically using physics (curvature).

### Stopping Criteria
1. Is the slope flat? Stop when the gradient norm is less than a threshold: $||\nabla f(x_k)|| < \epsilon$
2. Step size judgement. If update step $x_{k+1}$ and previous step $x_k$ almost overlap: $||x_{k+1} - x_k|| < \epsilon$
3. Function value judgement. $|f(x_{k+1} - f(x_k))| < \epsilon$
4. Time's up? Reach maximum iterations.

## Simple Relaxation (Coordinate Descent)

The deterministic version of Gibbs sampling. A method for nD problems. Only one variable moves at a time, others fixed.

### Algorithm Flow
Minimize $f(x_1, \dots, x_n)$.
- Pick $x^{(0)}$.
- Loop (until convergence):
    - Update $x_1$: Fix $x_2,\dots,x_n$, find $x_1$ minimizing $f$. $$x_1^{new} = argmin_{x_1}f(x_1, x_2^{old}, \dots, x_n^{old})$$ Since $f$ becomes a 1D function, we can use, f.e., the Newton's approach to solve for min.
- Update $x_2$: Fix $x_1^{new}, x_3, \dots, x_n$. Find $x_2^{new}$
- ...
- Update $x_n$: Fix all previous. Find $x_n^{new}$

## Steepest Descent

Steepest Descent is a **First-Order Optimization Algorithm**. Gradient ($\nabla f$): steepest uphill. Negative gradient ($-\nabla f$): steepest downhill.

The core: $$x_{k+1} = x_k - \alpha \nabla f(x_k)$$, where $\nabla f(x_k)$ gives the direction of the next move and $\alpha$ gives the step size (learning rate)

### Choosing $\alpha$ 
Using Line Search: $$\alpha_k = argmin_{\alpha > 0} f(x_k - \alpha \nabla f(x_k))$$

#TODO: add code examples to each approach